In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances
from src.models.similarity import build_distance_matrix, nearest
from src.data.player_ids import load_player_ids, display_name

df = load_all_snapshots(seasons=[2024])
pa = plate_appearances(df)
pitcher_ids = set(df["pitcher"].dropna().unique())

# Only metrics that a KBO box score also provides.
# No plate discipline, no batted-ball tracking.
ends = pa[pa["events"].notna() & ~pa["batter"].isin(pitcher_ids)]

HITS = {"single", "double", "triple", "home_run"}
NON_AB = {"walk", "intent_walk", "hit_by_pitch", "sac_fly",
          "sac_bunt", "catcher_interf", "truncated_pa"}

g = ends.groupby("batter")["events"]
box = pd.DataFrame({
    "pa": g.size(),
    "k": g.apply(lambda s: (s == "strikeout").sum()),
    "bb": g.apply(lambda s: (s == "walk").sum()),
    "hits": g.apply(lambda s: s.isin(HITS).sum()),
    "doubles": g.apply(lambda s: (s == "double").sum()),
    "triples": g.apply(lambda s: (s == "triple").sum()),
    "hr": g.apply(lambda s: (s == "home_run").sum()),
    "ab": g.apply(lambda s: (~s.isin(NON_AB)).sum()),
})

box["k_pct"] = box["k"] / box["pa"]
box["bb_pct"] = box["bb"] / box["pa"]
box["avg"] = box["hits"] / box["ab"]
box["slg"] = (box["hits"] + box["doubles"] + 2 * box["triples"]
              + 3 * box["hr"]) / box["ab"]
box["iso"] = box["slg"] - box["avg"]

box = box[box["pa"] >= 300]
print(f"{len(box)} batters with 300+ PA")
print(box[["k_pct", "bb_pct", "avg", "iso"]].describe().round(3).to_string())

270 batters with 300+ PA
         k_pct   bb_pct      avg      iso
count  270.000  270.000  270.000  270.000
mean     0.217    0.079    0.249    0.162
std      0.057    0.026    0.028    0.050
min      0.043    0.025    0.169    0.053
25%      0.172    0.061    0.231    0.124
50%      0.212    0.076    0.248    0.156
75%      0.258    0.098    0.268    0.194
max      0.397    0.178    0.332    0.379


In [2]:
KBO_FEATURES = ["k_pct", "bb_pct", "avg", "iso"]

print(box[KBO_FEATURES].corr().round(2).to_string())
print()

dist_kbo = build_distance_matrix(box, features=KBO_FEATURES)

ids = load_player_ids(box.index.tolist())
box = box.join(display_name(ids))

JUDGE = 592450
print("=== Judge, box-score space (KBO-compatible) ===")
out = nearest(JUDGE, dist_kbo, box, n=8, features=KBO_FEATURES)
out.insert(0, "name", box.loc[out.index, "name"])
print(out[["name", "distance"] + KBO_FEATURES].round(3).to_string(index=False))

        k_pct  bb_pct   avg   iso
k_pct    1.00    0.15 -0.46  0.29
bb_pct   0.15    1.00 -0.04  0.34
avg     -0.46   -0.04  1.00  0.28
iso      0.29    0.34  0.28  1.00

=== Judge, box-score space (KBO-compatible) ===
             name  distance  k_pct  bb_pct   avg   iso
     Tucker, Kyle     2.527  0.156   0.156 0.289 0.296
   Ohtani, Shohei     2.547  0.225   0.099 0.310 0.342
       Soto, Juan     2.726  0.167   0.178 0.288 0.281
   Ozuna, Marcell     3.467  0.246   0.104 0.304 0.247
     Marte, Ketel     3.584  0.178   0.099 0.292 0.268
    Pederson, Joc     3.637  0.234   0.118 0.275 0.240
Henderson, Gunnar     3.661  0.221   0.107 0.281 0.248
    Rooker, Brent     3.745  0.288   0.089 0.293 0.269


In [3]:
common = box.index.intersection(
    build_distance_matrix(box, features=KBO_FEATURES).index)

# Compare neighbour sets across the two spaces for many players
from src.models.batter_score import build_profiles, qualified
prof = qualified(build_profiles(df))
shared = prof.index.intersection(box.index)

dist_sc = build_distance_matrix(prof.loc[shared])
dist_bx = build_distance_matrix(box.loc[shared], features=KBO_FEATURES)

overlaps = []
for pid in shared:
    a = set(dist_sc.loc[pid].drop(pid).nsmallest(8).index)
    b = set(dist_bx.loc[pid].drop(pid).nsmallest(8).index)
    overlaps.append(len(a & b))

print(f"{len(shared)} batters in both spaces")
print(f"mean neighbour overlap: {np.mean(overlaps):.2f}/8")
print(pd.Series(overlaps).value_counts().sort_index().to_string())

270 batters in both spaces
mean neighbour overlap: 1.12/8
0    89
1    97
2    54
3    25
4     4
6     1
